# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedali3ff/Flyrank-internship-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [9]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Ahmedali3ff/Flyrank-internship-.git"
REPO_DIR = Path("/content/Flyrank-internship-")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

    print("Repo ready at:", REPO_DIR)
    print("Exists:", REPO_DIR.exists())

In [10]:
from pathlib import Path

DATA_PATH = Path("/content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

if DATA_PATH.exists():
    print("Dataset size:", DATA_PATH.stat().st_size / (1024 * 1024), "MB")

Dataset exists: True
Dataset path: /content/Flyrank-internship-/data/raw/content_refresh_anonymized.csv
Dataset size: 6.416006088256836 MB


In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shape:", df.shape)

Rows: 30000
Columns: 44
Shape: (30000, 44)


In [12]:
print("Available columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

Available columns:
01. content_id
02. client_id
03. search_volume
04. competition
05. competition_level
06. cpc
07. content_type
08. main_intent
09. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [24]:
print("SIGNAL 1 - STALENESS")
print("=" * 60)

# Bucket content age into interpretable freshness ranges
staleness_table = (
    df.groupby("staleness_bucket", observed=False)
          .size()
                .reset_index(name="n")
                )

display(staleness_table)

SIGNAL 1 - STALENESS


,staleness_bucket,n
0,"(-inf, 30.0]",20480
1,"(30.0, 90.0]",175
2,"(90.0, 180.0]",9171
3,"(180.0, 365.0]",169
4,"(365.0, inf]",5


In [28]:
print("SIGNAL 2 - SEARCH VOLUME")
print("=" * 60)

# Create volume buckets using quartiles
volume_table = (
    df.assign(
            volume_bucket=pd.qcut(
                        df["search_volume"],
                                    q=4,
                                                duplicates="drop"
                                                        )
                                                            )
                                                                .groupby("volume_bucket", observed=False)
                                                                    .size()
                                                                        .reset_index(name="n")
                                                                        )

display(volume_table)

SIGNAL 2 - SEARCH VOLUME


,volume_bucket,n
0,"(-0.001, 10.0]",18392
1,"(10.0, 20.0]",2290
2,"(20.0, 74000.0]",6850


In [29]:
print("SIGNAL VERDICTS")
print("=" * 60)

print("Signal 1 - Staleness: NEEDS OUTCOME CHECK")
print("Signal 2 - Search Volume: NEEDS OUTCOME CHECK")
print()
print("Reason:")
print("- Bucket counts show the signal distribution.")
print("- A verdict requires checking whether the signal is associated")
print("  with the action/flag we are trying to prioritize.")
print("- No future-window or label-derived input is used here.")

SIGNAL VERDICTS
Signal 1 - Staleness: NEEDS OUTCOME CHECK
Signal 2 - Search Volume: NEEDS OUTCOME CHECK

Reason:
- Bucket counts show the signal distribution.
- A verdict requires checking whether the signal is associated
  with the action/flag we are trying to prioritize.
- No future-window or label-derived input is used here.


In [31]:
print("SIGNAL ASSOCIATION CHECK")
print("=" * 60)

# Show available columns that may represent FlyRank flags/actions
flag_candidates = [
    col for col in df.columns
        if any(term in col.lower() for term in [
                "flag", "refresh", "ctr", "quick", "action"
                    ])
                    ]
print("Potential flag/action columns:")
for col in flag_candidates:
                        print(f"- {col}")

SIGNAL ASSOCIATION CHECK
Potential flag/action columns:
- ctr


In [32]:
print("CTR COLUMN CHECK")
print("=" * 60)

print("CTR dtype:", df["ctr"].dtype)
print("CTR non-null:", df["ctr"].notna().sum())
print("CTR unique values:", df["ctr"].nunique())

display(df[["ctr"]].describe())

CTR COLUMN CHECK
CTR dtype: float64
CTR non-null: 30000
CTR unique values: 401


,ctr
count,30000.000000
mean,0.510733
std,3.279162
min,0.000000
25%,0.000000
50%,0.070000
75%,0.290000
max,100.000000


In [34]:
print("PERFORMANCE SIGNAL CHECK")
print("=" * 60)

# Look for columns useful for the CTR-vs-position signal
performance_candidates = [
    col for col in df.columns
        if any(term in col.lower() for term in [
                "position",
                        "rank",
                                "impression",
                                        "click",
                                                "ctr"
                                                    ])
                                                    ]

for col in performance_candidates:
                                                        print(f"- {col}")

PERFORMANCE SIGNAL CHECK
- impressions_90d
- clicks_90d
- days_with_impressions
- impressions_last_30d
- clicks_last_30d
- impressions_prev_30d
- clicks_prev_30d
- ctr
- avg_position
- impression_tier
- position_tier


In [37]:
print("SIGNAL 2 - CTR VS POSITION")
print("=" * 60)

# Create interpretable position buckets
df["position_bucket"] = pd.cut(
    df["avg_position"],
        bins=[-float("inf"), 3, 10, 20, float("inf")],
            labels=["Top 3", "4-10", "11-20", "21+"]
            )

ctr_position_table = (
                df.groupby("position_bucket", observed=False)["ctr"]
                      .agg(["count", "mean", "median"])
                            .reset_index()
                                  .rename(columns={
                                            "count": "n",
                                                      "mean": "mean_ctr",
                                                                "median": "median_ctr"
                                                                      })
                                                                      )
display(ctr_position_table)

SIGNAL 2 - CTR VS POSITION


,position_bucket,n,mean_ctr,median_ctr
0,Top 3,2346,1.472869,0.00
1,4-10,11842,0.651045,0.16
2,11-20,7273,0.323443,0.10
3,21+,8539,0.211333,0.00


In [39]:
print("SIGNAL VERDICTS")
print("=" * 60)

print("Signal 1 - Staleness: MIXED")
print(
    "Reason: Most content is fresh (<=30 days), while a substantial "
        "share is 90-180 days old. Staleness is useful for identifying "
            "refresh candidates, but the distribution alone does not establish "
                "that older content should always be prioritized."
                )

print()

print("Signal 2 - CTR vs Position: CONFIRMED")
print(
                    "Reason: Mean CTR decreases consistently as average position worsens "
                        "(Top 3 > 4-10 > 11-20 > 21+). This supports the expected relationship "
                            "between search position and CTR."
                            )

SIGNAL VERDICTS
Signal 1 - Staleness: MIXED
Reason: Most content is fresh (<=30 days), while a substantial share is 90-180 days old. Staleness is useful for identifying refresh candidates, but the distribution alone does not establish that older content should always be prioritized.

Signal 2 - CTR vs Position: CONFIRMED
Reason: Mean CTR decreases consistently as average position worsens (Top 3 > 4-10 > 11-20 > 21+). This supports the expected relationship between search position and CTR.


In [42]:
print("BASELINE ACTION RULE")
print("=" * 60)

# Staleness points
staleness_score_map = {
    "(-inf, 30.0]": 0,
        "(30.0, 90.0]": 1,
            "(90.0, 180.0]": 2,
                "(180.0, 365.0]": 3,
                    "(365.0, inf]": 4,
                    }

                    # Position points
position_score_map = {
                        "Top 3": 0,
                            "4-10": 1,
                                "11-20": 2,
                                    "21+": 3,
                                    }
df["staleness_score"] = (
                                        df["staleness_bucket"]
                                            .astype(str)
                                                .map(staleness_score_map)
                                                    .fillna(0)
                                                    )
df["position_score"] = (
                                                        df["position_bucket"]
                                                            .astype(str)
                                                                .map(position_score_map)
                                                                    .fillna(0)
                                                                    )

                                                                    # ONE baseline score
df["baseline_score"] = (
                                                                        df["staleness_score"] +
                                                                            df["position_score"]
                                                                            )

                                                                            # One reason code
df["reason_code"] = "STALE_AND_LOW_POSITION"

                                                                            # Action label
df["action_label"] = pd.cut(
                                                                                df["baseline_score"],
                                                                                    bins=[-1, 1, 3, 7],
                                                                                        labels=["MONITOR", "REVIEW", "REFRESH"]
                                                                                        )
print("Rule:")
print("baseline_score = staleness_score + position_score")
print("reason_code = STALE_AND_LOW_POSITION")
print("action_label = MONITOR / REVIEW / REFRESH")
display(
                                                                                            df[
                                                                                                    [
                                                                                                                "baseline_score",
                                                                                                                            "reason_code",
                                                                                                                                        "action_label"
                                                                                                                                                ]
                                                                                                                                                    ].head(10)
                                                                                                                                                    )

BASELINE ACTION RULE
Rule:
baseline_score = staleness_score + position_score
reason_code = STALE_AND_LOW_POSITION
action_label = MONITOR / REVIEW / REFRESH


,baseline_score,reason_code,action_label
0,2,STALE_AND_LOW_POSITION,REVIEW
1,3,STALE_AND_LOW_POSITION,REVIEW
2,3,STALE_AND_LOW_POSITION,REVIEW
3,1,STALE_AND_LOW_POSITION,MONITOR
4,3,STALE_AND_LOW_POSITION,REVIEW
5,1,STALE_AND_LOW_POSITION,MONITOR
6,1,STALE_AND_LOW_POSITION,MONITOR
7,3,STALE_AND_LOW_POSITION,REVIEW
8,3,STALE_AND_LOW_POSITION,REVIEW
9,3,STALE_AND_LOW_POSITION,REVIEW


In [46]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("Output directory ready:")
print(os.path.exists("work/outputs"))

Output directory ready:
True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [47]:
print("RANKED BASELINE QUEUE")
print("=" * 60)

# Rank highest-priority items first
queue_columns = [
    "content_id",
        "baseline_score",
            "reason_code",
                "action_label",
                    "staleness_bucket",
                        "position_bucket",
                            "ctr",
                                "avg_position",
                                    "search_volume",
                                    ]

                                    # Keep only columns that exist in the current dataframe
queue_columns = [col for col in queue_columns if col in df.columns]

baseline_queue = (
                                        df[queue_columns]
                                            .sort_values(
                                                    by=["baseline_score", "avg_position"],
                                                            ascending=[False, False]
                                                                )
                                                                    .reset_index(drop=True)
                                                                    )

                                                                    # Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(
                                                                        output_path,
                                                                            index=False
                                                                            )

print(f"Queue written to: {output_path}")
print(f"Rows: {len(baseline_queue):,}")

display(baseline_queue.head(10))

RANKED BASELINE QUEUE
Queue written to: work/outputs/baseline_action_score.csv
Rows: 30,000


,content_id,baseline_score,reason_code,action_label,staleness_bucket,position_bucket,ctr,avg_position,search_volume
0,content_8d56efff1e71,7,STALE_AND_LOW_POSITION,REFRESH,"(365.0, inf]",21+,0.00,35.0,0.0
1,content_f6fdf87348f6,7,STALE_AND_LOW_POSITION,REFRESH,"(365.0, inf]",21+,0.00,32.5,0.0
2,content_6476d1d8c050,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,67.8,10.0
3,content_7a888d3d99c8,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,67.6,90.0
4,content_15fe075b97bc,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,67.0,NaN
5,content_d25a099b3726,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,64.5,10.0
6,content_074ba6ead17b,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,48.0,0.0
7,content_dd413158df3c,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,46.1,20.0
8,content_afd26a07382d,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.00,45.1,10.0
9,content_5feee3994adb,6,STALE_AND_LOW_POSITION,REFRESH,"(180.0, 365.0]",21+,0.01,39.0,0.0


In [50]:
print("TOP-10 REVIEW")
print("=" * 60)

top10_review = baseline_queue.head(10).copy()

for idx, row in top10_review.iterrows():
    print(f"\n#{idx+1}")
    print(f"Content ID: {row['content_id']}")
    print(f"Action: {row['action_label']}")
    print(
                        f"Why it's here: Score={row['baseline_score']} "
                                f"because of {row['reason_code']} "
                                        f"(Staleness={row['staleness_bucket']}, "
                                                f"Position={row['position_bucket']})."
                                                    )
    print(
                                                    "What would make it wrong: "
                                                                        "The content may still be accurate and useful despite "
                                                                                "its age or low position, or the position data may not "
                                                                                        "represent true improvement opportunity."
                                                                                            )

TOP-10 REVIEW

#1
Content ID: content_8d56efff1e71
Action: REFRESH
Why it's here: Score=7 because of STALE_AND_LOW_POSITION (Staleness=(365.0, inf], Position=21+).
What would make it wrong: The content may still be accurate and useful despite its age or low position, or the position data may not represent true improvement opportunity.

#2
Content ID: content_f6fdf87348f6
Action: REFRESH
Why it's here: Score=7 because of STALE_AND_LOW_POSITION (Staleness=(365.0, inf], Position=21+).
What would make it wrong: The content may still be accurate and useful despite its age or low position, or the position data may not represent true improvement opportunity.

#3
Content ID: content_6476d1d8c050
Action: REFRESH
Why it's here: Score=6 because of STALE_AND_LOW_POSITION (Staleness=(180.0, 365.0], Position=21+).
What would make it wrong: The content may still be accurate and useful despite its age or low position, or the position data may not represent true improvement opportunity.

#4
Content ID:

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.